<a href="https://colab.research.google.com/github/sherly2212/sql-ecommerce-analysis/blob/main/sql_ecommerce_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Persiapan Data dan Database

In [27]:
import pandas as pd
import sqlite3

interaksi = pd.read_csv("E-commerece sales data 2024.csv")
produk = pd.read_csv("product_details.csv")

In [28]:
produk = pd.read_csv("product_details.csv", on_bad_lines="skip")

In [29]:
print(produk.shape)

(10002, 28)


In [30]:
import pandas as pd
import sqlite3

interaksi = pd.read_csv("E-commerece sales data 2024.csv")
produk = pd.read_csv("product_details.csv")

In [31]:
produk = pd.read_csv("product_details.csv", engine="python", on_bad_lines="skip")
print(produk.shape)

(10002, 28)


In [32]:
import pandas as pd
import sqlite3

interaksi = pd.read_csv("E-commerece sales data 2024.csv")
produk = pd.read_csv("product_details.csv", engine="python", on_bad_lines="skip")
print(interaksi.shape, produk.shape)

(3294, 5) (10002, 28)


In [33]:
produk = pd.read_csv("product_details.csv", on_bad_lines="skip")
print(produk.shape)

(10002, 28)


In [34]:
produk = pd.read_csv("product_details.csv", encoding="latin-1")
print(produk.shape)

(10002, 28)


In [35]:
pd.read_sql("""
SELECT COUNT(*) AS tanpa_produk
FROM interaksi i
LEFT JOIN produk p ON i.product_id = p.product_id
WHERE p.product_id IS NULL
""", conn)

,tanpa_produk
0,0


In [36]:
produk = pd.read_csv("product_details.csv", encoding="latin-1")

In [37]:
# 1. Cleaning interaksi
interaksi = interaksi.dropna(how="all").drop(columns=["Unnamed: 4"])
interaksi.columns = ["user_id", "product_id", "interaction_type", "timestamp"]
interaksi["user_id"] = interaksi["user_id"].astype(int)
interaksi["timestamp"] = pd.to_datetime(interaksi["timestamp"], dayfirst=True).dt.strftime("%Y-%m-%d")

# 2. Cleaning produk
produk_sql = produk[["Uniqe Id", "Product Name", "Category", "Selling Price"]].copy()
produk_sql.columns = ["product_id", "product_name", "category", "price"]
produk_sql["price"] = pd.to_numeric(
    produk_sql["price"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False),
    errors="coerce"
)

# 3. Buat database dan isi tabel
conn = sqlite3.connect("ecommerce.db")
interaksi.to_sql("interaksi", conn, index=False, if_exists="replace")
produk_sql.to_sql("produk", conn, index=False, if_exists="replace")

pd.read_sql("SELECT COUNT(*) AS jumlah_baris FROM interaksi", conn)

,jumlah_baris
0,2999


In [38]:
pd.read_sql("""
SELECT COUNT(*) AS tanpa_produk
FROM interaksi i
LEFT JOIN produk p ON i.product_id = p.product_id
WHERE p.product_id IS NULL
""", conn)

,tanpa_produk
0,0


## 2. Query 1: Proporsi Jenis Interaksi

In [39]:
pd.read_sql("""
SELECT
    interaction_type,
    COUNT(*) AS jumlah,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS persen
FROM interaksi
WHERE interaction_type IS NOT NULL
GROUP BY interaction_type
ORDER BY jumlah DESC
""", conn)

,interaction_type,jumlah,persen
0,like,1145,39.9
1,view,871,30.3
2,purchase,855,29.8


## 3. Query 2: Subkategori dengan Porsi Purchase Tertinggi

In [40]:
pd.read_sql("""
SELECT
    TRIM(SUBSTR(
        SUBSTR(p.category, INSTR(p.category, '|') + 1),
        1,
        INSTR(SUBSTR(p.category, INSTR(p.category, '|') + 1), '|') - 1
    )) AS sub_category,
    COUNT(*) AS total,
    SUM(CASE WHEN i.interaction_type = 'purchase' THEN 1 ELSE 0 END) AS purchase,
    ROUND(SUM(CASE WHEN i.interaction_type = 'purchase' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS persen_purchase
FROM interaksi i
JOIN produk p ON i.product_id = p.product_id
WHERE i.interaction_type IS NOT NULL
  AND p.category IS NOT NULL
GROUP BY sub_category
HAVING total >= 20
ORDER BY persen_purchase DESC
LIMIT 10
""", conn)

,sub_category,total,purchase,persen_purchase
0,Office & School Supplies,26,11,42.3
1,Dolls & Accessories,108,42,38.9
2,Dress Up & Pretend Play,150,52,34.7
3,Stuffed Animals & Plush Toys,117,40,34.2
4,Learning & Education,88,30,34.1
5,Hobbies,176,59,33.5
6,Bedding,48,16,33.3
7,,321,99,30.8
8,Play Vehicles,63,19,30.2
9,Outdoor Recreation,60,18,30.0


## 4. Query 3: Tren Bulanan

In [41]:
pd.read_sql("""
SELECT
    STRFTIME('%Y-%m', timestamp) AS bulan,
    COUNT(*) AS total,
    SUM(CASE WHEN interaction_type = 'view' THEN 1 ELSE 0 END) AS view,
    SUM(CASE WHEN interaction_type = 'like' THEN 1 ELSE 0 END) AS like_,
    SUM(CASE WHEN interaction_type = 'purchase' THEN 1 ELSE 0 END) AS purchase
FROM interaksi
WHERE interaction_type IS NOT NULL
GROUP BY bulan
ORDER BY bulan
""", conn)

,bulan,total,view,like_,purchase
0,2022-12,72,21,26,25
1,2023-01,415,124,169,122
2,2023-02,135,43,54,38
3,2023-03,122,37,51,34
4,2023-04,118,36,45,37
5,2023-05,124,41,46,37
6,2023-06,120,36,47,37
7,2023-07,106,32,46,28
8,2023-08,93,30,37,26
9,2023-09,88,24,33,31


## 5. Query 4: Peringkat Bulan (CTE dan RANK)

In [42]:
pd.read_sql("""
WITH bulanan AS (
    SELECT
        STRFTIME('%Y-%m', timestamp) AS bulan,
        COUNT(*) AS total
    FROM interaksi
    WHERE interaction_type IS NOT NULL
    GROUP BY bulan
)
SELECT
    bulan,
    total,
    RANK() OVER (ORDER BY total DESC) AS peringkat
FROM bulanan
ORDER BY peringkat
LIMIT 5
""", conn)

,bulan,total,peringkat
0,2023-11,565,1
1,2023-12,504,2
2,2023-01,415,3
3,2023-10,409,4
4,2023-02,135,5


In [43]:
interaksi = pd.read_csv("E-commerece sales data 2024.csv")
produk = pd.read_csv("product_details.csv", encoding="latin-1")